# MVP sobre datos reales (anonimizados) — Mesa de servicio

Carga `incidentes_anonimizado.db` desde Google Drive, hace el análisis (EDA + sentimiento + frustración + categorías) y genera `resultados.json` para el dashboard.

**Flujo:** los datos viven en tu Drive (privados); este notebook (código) vive en GitHub. Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Conectar Drive y localizar la base

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
FILE_ID = '1FZhxpcYeKsriLIHpKSJlfZNjygXZXqmX'   # id del .db en Drive (respaldo)
cand = glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True)
if cand:
    RUTA_DB = cand[0]; print('Encontrado en Drive:', RUTA_DB)
else:
    import gdown; RUTA_DB = '/content/incidentes_anonimizado.db'
    gdown.download(id=FILE_ID, output=RUTA_DB, quiet=False)

## 2. Cargar los datos (detecta la tabla con `resumen`)

In [ ]:
import sqlite3, pandas as pd, numpy as np, re, json, os
con = sqlite3.connect(RUTA_DB)

tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)['name'].tolist()
print('Tablas:', tablas)

TABLA = None
for t in tablas:
    cols = pd.read_sql('SELECT * FROM "%s" LIMIT 1' % t, con).columns.str.lower().tolist()
    if 'resumen' in cols:
        TABLA = t; break
TABLA = TABLA or (tablas[0] if tablas else None)
print('Tabla usada:', TABLA)

df = pd.read_sql('SELECT * FROM "%s"' % TABLA, con)
df.columns = df.columns.str.lower()
print('Filas, columnas:', df.shape)
df.head()

## 3. Limpieza básica

In [ ]:
COL_TEXTO = 'resumen' if 'resumen' in df.columns else df.select_dtypes('object').columns[0]
COL_LABEL = 'clasificacion' if 'clasificacion' in df.columns else None

df = df.drop_duplicates()
df[COL_TEXTO] = df[COL_TEXTO].astype(str).str.strip()
print('Texto:', COL_TEXTO, '| Etiqueta:', COL_LABEL)
print('Tras limpieza:', df.shape)

## 4. Análisis exploratorio (categorías)

In [ ]:
import matplotlib.pyplot as plt
if COL_LABEL:
    top = df[COL_LABEL].value_counts().head(10)
    display(top)
    top[::-1].plot(kind='barh'); plt.title('Top categorías'); plt.tight_layout(); plt.show()

## 5. Sentimiento y nivel de frustración
Léxico simple en español adaptado a tickets técnicos (MVP).

In [ ]:
POS = set('gracias ok correcto solucionado resuelto disponible exitoso'.split())
NEG = set('error falla fallo sin no cancelacion cancelación pendiente caido caída bloqueo rechazo duplicado incidencia problema demora retraso urgente'.split())

def sent(t):
    palabras = re.findall(r'[a-záéíóúñ]+', str(t).lower())
    p = sum(w in POS for w in palabras); n = sum(w in NEG for w in palabras)
    return round((p - n) / (p + n), 3) if (p + n) else 0.0

df['sentimiento'] = df[COL_TEXTO].apply(sent)
df['frustracion'] = ((1 - df['sentimiento']) / 2 * 100).round(1)
df['banda'] = df['frustracion'].apply(lambda f: 'alto' if f >= 66 else ('medio' if f >= 40 else 'bajo'))
display(df[['sentimiento','frustracion']].describe())
df['banda'].value_counts().reindex(['bajo','medio','alto']).plot(kind='bar', color=['#22c55e','#f59e0b','#ef4444'])
plt.title('Distribución de frustración'); plt.tight_layout(); plt.show()

## 6. Generar `resultados.json` para el dashboard

In [ ]:
from datetime import datetime

bins = [-1, -0.6, -0.2, 0.2, 0.6, 1.0001]
etq = ['muy negativo','negativo','neutro','positivo','muy positivo']
hist = pd.cut(df['sentimiento'], bins=bins, labels=etq, include_lowest=True).value_counts().reindex(etq).fillna(0).astype(int)

top_cat = df[COL_LABEL].value_counts().head(8).to_dict() if COL_LABEL else {}

alta = df[df['banda'] == 'alto']
if COL_LABEL and len(alta):
    imp = alta[COL_LABEL].value_counts(normalize=True).head(6)
    impulsores = [{'variable': str(k), 'importancia': round(float(v), 3)} for k, v in imp.items()]
else:
    impulsores = []

id_col = 'orden_id' if 'orden_id' in df.columns else df.columns[0]
def recom(f):
    return 'Contacto inmediato y plan de retencion' if f >= 66 else ('Seguimiento proactivo' if f >= 40 else 'Monitoreo estandar')
altos = df.sort_values('frustracion', ascending=False).head(15)
tickets = [{
    'id': str(r[id_col]), 'cliente': str(r[id_col]),
    'clasificacion': str(r[COL_LABEL]) if COL_LABEL else '',
    'frustracion': float(r['frustracion']), 'riesgo': float(r['frustracion']),
    'recomendacion': recom(r['frustracion'])
} for _, r in altos.iterrows()]

resultados = {
    'generado': datetime.now().strftime('%Y-%m-%d %H:%M') + ' (datos reales anonimizados)',
    'kpis': {
        'total_tickets': int(len(df)),
        'phishing_aislados': 0,
        'procesados': int(len(df)),
        'pct_alto_riesgo': round(float((df['banda']=='alto').mean()*100), 1),
        'frustracion_promedio': round(float(df['frustracion'].mean()), 1),
        'sentimiento_promedio': round(float(df['sentimiento'].mean()), 3)
    },
    'clasificacion': {str(k): int(v) for k, v in top_cat.items()},
    'distribucion_riesgo': {k: int(v) for k, v in df['banda'].value_counts().reindex(['bajo','medio','alto']).fillna(0).astype(int).items()},
    'sentimiento_hist': {'bins': etq, 'conteo': hist.tolist()},
    'impulsores': impulsores,
    'tickets_alto_riesgo': tickets
}

with open('resultados.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print('Generado resultados.json')
print(json.dumps(resultados['kpis'], indent=2, ensure_ascii=False))

## 7. Descargar `resultados.json`
Descárgalo y súbelo al repo de GitHub en `docs/data/resultados.json` (reemplazando el de muestra). El dashboard se actualiza solo.

In [ ]:
try:
    from google.colab import files
    files.download('resultados.json')
except Exception as e:
    print('Descarga manual del archivo resultados.json. Detalle:', e)